# PV + battery energy storage

A standalone PNM-data demonstration: load data, declare the PV/BES assets and economics, dispatch, calculate LCOE, plot, and run resilience cases.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
while not (root / 'enliten').is_dir():
    if root.parent == root: raise RuntimeError('Run from inside the ENLITEN repository.')
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from enliten import ChargingPath, Generation, LCOECalculator, Site, Storage, System
data_dir = root / 'examples' / 'data'

def profile(filename):
    frame = pd.read_csv(data_dir / filename)
    return pd.Series(frame['PNM'].to_numpy(float), index=pd.to_datetime(frame.iloc[:, 0], utc=True))

demand_full = profile('PNM_demand.csv')
pv_full = profile('PNM_pv_ac_1MW_av.csv')
assert demand_full.index.equals(pv_full.index)

hours, start = 24 * 14, pd.Timestamp('2023-07-01', tz='UTC')
window = demand_full.index.get_loc(start)
load = (demand_full.iloc[window:window + hours] * 0.05).rename('load_MW')
pv_multiplier, bes_capacity_MWh, bes_power_MW = 3_000.0, 750.0, 150.0
pv_capex = pv_full.max() * pv_multiplier * 1_000 * 1_430
pv_opex = pv_full.max() * pv_multiplier * 1_000 * 24
bes_capex = bes_capacity_MWh * 1_000 * 300

site = Site('microgrid')
pv = Generation('pv', site, pv_full.iloc[window:window + hours] * pv_multiplier, 'electric', capex=pv_capex, opex=pv_opex)
bes = Storage('bes', site, bes_capacity_MWh, bes_power_MW, 'electric', 'electric', 0.90,
              maximum_stored_energy_rate_MW=bes_power_MW, capex=bes_capex, opex=0.025 * bes_capex)
path = ChargingPath('pv', 'bes', 'electric', 'electric', 0.90, bes_power_MW / 0.90)
system = System(load, [bes, pv], [path])
system.timeseries.head()

In [ ]:
system.operation_metrics()

In [ ]:
LCOECalculator.from_system(system).calculate_lcoe_metrics()

In [ ]:
fig, ax = system.timeseries_plot_source(start_date=0, days=2)
fig

In [ ]:
fig, ax = system.timeseries_plot_group(start_date=0, days=2)
fig

In [ ]:
fig, ax = system.plot_storage_capacity(start_date=0, days=2)
fig

In [ ]:
cases = system.resilience_cases(critical_load_MW=20.0, target_hours=24, n_starts=20, seed=7)
display(cases)
system.resilience_summary